In [1]:
!pip install sentence-transformers

In [8]:
from sentence_transformers import SentenceTransformer
import pandas as pd, numpy as np

df = pd.read_csv('cleaned_data.csv')
# model = SentenceTransformer('all-MiniLM-L6-v2')
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

embeddings = model.encode(df['description_clean'].tolist(), show_progress_bar=True)
np.save('club_embeddings.npy', embeddings)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

In [9]:
from sklearn.metrics.pairwise import cosine_similarity

def recommend(user_text, top_n=5, category_filter=None):
    query_embedding = model.encode([user_text])
    scores = cosine_similarity(query_embedding, embeddings)[0]
    df['score'] = scores

    result = df.copy()
    if category_filter:
        result = result[result['category'].isin(category_filter)]

    return result.nlargest(top_n, 'score')[['name','category','description_clean','score','telegram_url','instagram_url']]

In [10]:
def normalize_query(user_text):


SyntaxError: incomplete input (2881864512.py, line 2)

In [33]:
# USER_QUERY = ""
# USER_QUERY = normalize_query(USER_QUERY)
recommend("i like programming and wanna participate in hackathons")

,name,category,description,score,telegram_url,instagram_url
55,Not Threat But Thread (NTBT),social,Our club aims to create a supporting and welco...,0.375684,https://t.me/ntbt,https://www.instagram.com/nu_ntbt/
96,Rave Club,cultural,Our goal is to connect people interested in DJ...,0.362407,https://t.me/rave,https://www.instagram.com/rave.nu/
65,NU Data Science Club,professional,DataSci Club is the community for anyone passi...,0.355275,https://t.me/nu_datasci,https://www.instagram.com/nu_datasci/
73,NU Mafia,social,A community of Mafia game lovers who don't jus...,0.348642,https://t.me/nu_polemica,https://www.instagram.com/nu_mafia_club/
35,Game Development Club,art,NU Game Development Club fosters a community o...,0.328468,https://t.me/nu_gdc,https://www.instagram.com/nu_gdc/


In [27]:
print(df[df['name'] == "NU Shyraq"])

    Unnamed: 0  id       name  category               email  \
81         103  55  NU Shyraq  cultural  nushyraq@nu.edu.kz   

                                          description established  \
81  Shyraq Club is Nazarbayev University's officia...  2020-09-01   

             telegram_url                         instagram_url  \
81  https://t.me/nushyraq  https://www.instagram.com/nu.shyraq/   

                                    description_clean    score  
81  Shyraq Club is Nazarbayev University's officia...  0.55308  


In [32]:
# # test_similarity.py — run this today
# from sentence_transformers import SentenceTransformer
# from sklearn.metrics.pairwise import cosine_similarity
# import pandas as pd, numpy as np

# df = pd.read_csv('updated_clubs.csv')
# model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
# embeddings = model.encode(df['description'].tolist())

# # test with raw input first
test_queries = [
    "я люблю футбол и хочу новых друзей",        # Russian
    "i like programming and participating in hackathons",   # Russian
    "I like photography and creative arts",        # English
]

for q in test_queries:
    vec = model.encode([q])
    scores = cosine_similarity(vec, embeddings)[0]
    df['score'] = scores
    top3 = df.nlargest(3, 'score')[['name','category','score']]
    print(f"\nQuery: {q}")
    print(top3.to_string())


Query: я люблю футбол и хочу новых друзей
                      name  category     score
87  NU Women Football Club    sports  0.509689
84         NU Table Tennis    sports  0.327770
96               Rave Club  cultural  0.323267

Query: i like programming and participating in hackathons
                            name  category     score
55  Not Threat But Thread (NTBT)    social  0.368872
73                      NU Mafia    social  0.351445
96                     Rave Club  cultural  0.341743

Query: I like photography and creative arts
                name  category     score
12    Art Revolution  cultural  0.424862
13        Art Studio  cultural  0.404143
0   25th July Cinema  cultural  0.388603
